In [ ]:
! pip install kaggle-environments --upgrade -q

In [ ]:
# Random sticky agent

# Check both own and opponent action to update number of times machine is pulled
# Use pulls to calculate overall decay of prior probability
# If decayed significantly add to blacklist

# If pulled more than 20 times and reward is low; > 40 times; add to yellow list
# Yellow list 1 and 2 based on higher and lower reward rates
# Reduced stickiness for yellow list
# x% chance that yellow list is excluded from bandit list


# If no reward, but opponent streak >= 3, pick opponent's bandit
# If no reward 3 times on opponent pick, blacklist bandit and move on
# If bandit picked > 10 times with no reward, blacklist it



# we need to understand who we are Player 1 or 2
# player = int(last_step == observation.lastActions[1])

# if last_reward > 0:
#     bandit_state[observation.lastActions[player]][0] += last_reward * step
# else:
#     bandit_state[observation.lastActions[player]][1] += step

In [ ]:
%%writefile agent.py

import random

def random_agent(observation, configuration):
    return random.randrange(configuration.banditCount)

In [ ]:
%%writefile randomStickyAgent.py

import random

oppStreak = 5
oppCopyBreak = 5
blankStreak = 15
stickyStreak = 5

decayHalfLife = 20
accWinRate = 0.1

blacklist = []
yellowlist = []
sticky = 0

bandit = 0

lastBandit = -1
totalReward = 0

banditSelectionsOvr = None
banditSelections = None
banditRewards = None
banditWinRate = None
banditBlanks = None


def random_sticky(obs, config):
    
    global oppStreak, oppCopyBreak, blankStreak, stickyStreak, decayHalfLife, accWinRate, blacklist, yellowlist, sticky, bandit, \
    lastBandit, totalReward, banditSelectionsOvr, banditSelections, banditRewards, banditWinRate, banditBlanks
    
    # Initialize metrics being monitored
    if obs.step == 0:
        banditSelectionsOvr = [0] * config["banditCount"]
        banditSelections = [0] * config["banditCount"]
        banditRewards = [0] * config["banditCount"]
        banditWinRate = [0] * config["banditCount"]
        banditBlanks = [0] * config["banditCount"]
        bandit = random.randrange(config.banditCount)
    
    # Increase blacklist rate
    if obs.step > 1500:
        blankStreak = 8
        stickyStreak = 3
        
    
    
    ############################
    # Previous action rewards
    ############################
    
    # After step 0
    if lastBandit > -1:
        # Calculate last action's reward
        stepReward = obs.reward - totalReward
        banditRewards[lastBandit] += stepReward
        totalReward += stepReward
        
        # No reward from last action
        if stepReward == 0:
            banditBlanks[lastBandit] += 1
            # If sticky active; push sticky
            if sticky > 0:
                sticky += 1
            if banditBlanks[lastBandit] >= blankStreak:
                blacklist.append(lastBandit)
        # Last action successful
        else:
            banditBlanks[lastBandit] = 0
            # If inactive; activate sticky
            if sticky == 0:
                sticky += 1
            # If active; linger
            elif sticky > 1:
                sticky -= 1
    
        # Find self and opponent
        moi = int(lastBandit == obs.lastActions[1])
        if moi == 0: opp = 1
        else: opp = 0

        # Update selection tally
        banditSelections[lastBandit] += 1
        banditSelectionsOvr[lastBandit] += 1
        banditSelectionsOvr[obs.lastActions[opp]] += 1

        # Calculate win rate
        for i in range(config.banditCount):
            if banditSelections[i] > 0:
                banditWinRate[i] =  banditRewards[i] / banditSelections[i]


        # Update blacklist
        for i in [x for x in list(range(config.banditCount)) if x not in blacklist]:
            if banditSelectionsOvr[i] > decayHalfLife*2 and banditWinRate[i] < accWinRate*2:
                blacklist.append(i)
            if banditSelectionsOvr[i] > decayHalfLife and banditWinRate[i] < accWinRate:
                blacklist.append(i)

        # Update yellow list
        for i in [x for x in list(range(config.banditCount)) if x not in blacklist and x not in yellowlist]:
            if banditSelectionsOvr[i] > decayHalfLife*2 and banditWinRate[i] < accWinRate*4:
                yellowlist.append(i)
            if banditSelectionsOvr[i] > decayHalfLife and banditWinRate[i] < accWinRate*2:
                yellowlist.append(i)

            if obs.step > 1500:
                if banditWinRate[i] < accWinRate*2 and i in yellowlist:
                    yellowlist.remove(i)
                    blacklist.append(i)




        #################
        # Choose action
        #################

        perc = random.randrange(100)

        # Stickiness over extended
        if sticky > stickyStreak:
            sticky = 0

        # If sticky not active
        if sticky == 0:

            # Explore
            if obs.step < 500:
                # Make random pick
                banditList = [x for x in list(range(config.banditCount)) if x not in blacklist]
                bandit = random.sample(banditList, 1)[0]

            # Exploit
            elif obs.step > 1500:
                maxwin = max(banditWinRate)
                bandit = banditWinRate.index(maxwin)

            else:
                if perc < 70:
                    # Make random pick
                    banditList = [x for x in list(range(config.banditCount)) if x not in blacklist]
                    bandit = random.sample(banditList, 1)[0]
                else:
                    # Make random pick
                    banditList = [x for x in list(range(config.banditCount)) if x not in blacklist and x not in yellowlist]
                    bandit = random.sample(banditList, 1)[0]



        # If sticky active
        # Choose previous
        else:
            bandit = lastBandit

    
    
    ##################
    # Take action
    ##################
    
    # Store chosen
    lastBandit = bandit
    
    print("Step: {}".format(obs.step))
    print("Length of yellowlist: {}".format(len(set(yellowlist))))
    print("Length of blacklist: {}".format(len(set(blacklist))))
    
    
    # Return chosen bandit
    return bandit


In [ ]:
# %%writefile cycle_bandit.py

# bandit = 0

# lastBandit = -1
# totalReward = 0

# banditSelections = None
# banditRewards = None

# def cycle_bandit(obs, config):
#     global bandit, lastBandit, totalReward, banditSelections, banditRewards

#     if obs.step == 0:
#         banditSelections = [0] * config["banditCount"]
#         banditRewards = [0] * config["banditCount"]

#     if lastBandit > -1:
#         stepReward = obs.reward - totalReward
#         banditRewards[lastBandit] += stepReward
#         totalReward += stepReward
    
#     lastBandit = bandit
#     return bandit
    
            
    

In [ ]:
from kaggle_environments import make

env = make("mab", debug=True)

env.run(["randomStickyAgent.py", "agent.py"])
env.render(mode="ipython", width=800, height=800)